## PHASE 1: Setup & Configuration

In [1]:
from dotenv import load_dotenv
import os

path = r'C:\Users\user\lca-lc-foundations\.env'
load_dotenv(dotenv_path = path) 

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [3]:
def clean_response(response):
    """Extract clean text from agent response"""
    last_msg = response['messages'][-1]
    
    if isinstance(last_msg.content, list):
        # Handle Gemini's list format
        for item in last_msg.content:
            if isinstance(item, dict) and item.get('type') == 'text':
                return item['text']
    else:
        return last_msg.content
    
    return "No text content found"



## PHASE 2: Tools Creation

In [4]:
from langchain.tools import tool 
from tavily import TavilyClient

tavily = TavilyClient()

@tool
def web_search(query: str) -> str: # -> are type hints ! and they are just DOCUMENTATION, not enfocement
                                   # Python does not enforce type hints, they're for develpers to read 
                                   # Necaissiarly and critical for agents 
    """Search the web for cooking recepies"""
    return tavily.search(query) 
response = web_search.invoke("What are te best recepies in the world")  #should use .invoke with tools ! 

In [5]:
print (response)

{'query': 'What are te best recepies in the world', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.amazon.com/Best-Recipes-World-Mark-Bittman/dp/0767906721', 'title': 'The Best Recipes in the World: Bittman, Mark - Amazon.com', 'content': 'A comprehensive global cookbook featuring over 1,000 accessible recipes from dozens of countries, including European, Asian, African, Middle Eastern, and', 'score': 0.99998605, 'raw_content': None}, {'url': 'https://www.youtube.com/watch?v=IOik-6yXto0', 'title': 'We Ranked the BEST Recipes in the World and the Results Will ...', 'content': "Inside the Kitchen is back with this week's top ten recipe list! These home cooks stepped up big time with mouthwatering recipes that just", 'score': 0.99994075, 'raw_content': None}, {'url': 'https://www.tasteatlas.com/best-recipes-in-the-world', 'title': 'Top 100 Traditional Recipes in the World - TasteAtlas', 'content': 'Top 100 Traditional Recipes in the World · Vor

In [6]:
from pprint import pprint 

pprint (response)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'What are te best recepies in the world',
 'request_id': '96b6e0ca-b5f1-4a04-918e-2c402c7b35ca',
 'response_time': 0.96,
 'results': [{'content': 'A comprehensive global cookbook featuring over 1,000 '
                         'accessible recipes from dozens of countries, '
                         'including European, Asian, African, Middle Eastern, '
                         'and',
              'raw_content': None,
              'score': 0.99998605,
              'title': 'The Best Recipes in the World: Bittman, Mark - '
                       'Amazon.com',
              'url': 'https://www.amazon.com/Best-Recipes-World-Mark-Bittman/dp/0767906721'},
             {'content': "Inside the Kitchen is back with this week's top ten "
                         'recipe list! These home cooks stepped up big time '
                         'with mouthwatering recipes that just',
              'raw_content': None,
         

In [7]:
import json
print(json.dumps(response, indent=2, ensure_ascii=False))

{
  "query": "What are te best recepies in the world",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.amazon.com/Best-Recipes-World-Mark-Bittman/dp/0767906721",
      "title": "The Best Recipes in the World: Bittman, Mark - Amazon.com",
      "content": "A comprehensive global cookbook featuring over 1,000 accessible recipes from dozens of countries, including European, Asian, African, Middle Eastern, and",
      "score": 0.99998605,
      "raw_content": null
    },
    {
      "url": "https://www.youtube.com/watch?v=IOik-6yXto0",
      "title": "We Ranked the BEST Recipes in the World and the Results Will ...",
      "content": "Inside the Kitchen is back with this week's top ten recipe list! These home cooks stepped up big time with mouthwatering recipes that just",
      "score": 0.99994075,
      "raw_content": null
    },
    {
      "url": "https://www.tasteatlas.com/best-recipes-in-the-world",
      "title": "Top 1

In [8]:
def show_full_response(response_dict):
    """Print the ENTIRE response with no truncation"""
    print(f"🔍 Query: {response_dict.get('query')}")
    print(f"⏱️ Response time: {response_dict.get('response_time')}s")
    print(f"\n📊 RESULTS ({len(response_dict.get('results', []))}):")
    print("-" * 50)
    
    for i, r in enumerate(response_dict.get('results', []), 1):
        print(f"\n{i}. {r.get('title')}")
        print(f"   URL: {r.get('url')}")
        print(f"   Content: {r.get('content')}")  # 👈 FULL content, no truncation
        print(f"   Score: {r.get('score')}")
        if r.get('raw_content'):
            print(f"   Raw: {r.get('raw_content')}")
    print("-" * 50)

# Use it
raw = web_search.invoke("best pasta recipes")
show_full_response(raw)

🔍 Query: best pasta recipes
⏱️ Response time: 0.73s

📊 RESULTS (5):
--------------------------------------------------

1. Easy Pasta Recipes | Best Homemade Pasta Dinner Ideas
   URL: https://therecipecritic.com/pasta-recipes/
   Content: # Pasta. ## Best Pasta Recipes & Pasta Combinations You Will Love! If you asked me to name some of my top favorite foods, pasta would definitely be up there at the top. I do not know what it is about it, but I just crave a good pasta, almost all the time! Whether it is just with some sauce and cheese, or with chicken and vegetables added in to it, you can never go wrong! If you or someone you know is on any dietary restrictions do not worry because there are plenty of pasta recipes that can still accommodate you! Check out the pasta recipes that interest you so the next time you get that craving you know just what to make! ## Creamy Garlic Ditalini Pasta. ## Creamy Sausage Spaghetti. ## Bang Bang Shrimp Pasta. ## Marry Me Stuffed Shells. ## Minestron

In [9]:
show_full_response(response)

🔍 Query: What are te best recepies in the world
⏱️ Response time: 0.96s

📊 RESULTS (5):
--------------------------------------------------

1. The Best Recipes in the World: Bittman, Mark - Amazon.com
   URL: https://www.amazon.com/Best-Recipes-World-Mark-Bittman/dp/0767906721
   Content: A comprehensive global cookbook featuring over 1,000 accessible recipes from dozens of countries, including European, Asian, African, Middle Eastern, and
   Score: 0.99998605

2. We Ranked the BEST Recipes in the World and the Results Will ...
   URL: https://www.youtube.com/watch?v=IOik-6yXto0
   Content: Inside the Kitchen is back with this week's top ten recipe list! These home cooks stepped up big time with mouthwatering recipes that just
   Score: 0.99994075

3. Top 100 Traditional Recipes in the World - TasteAtlas
   URL: https://www.tasteatlas.com/best-recipes-in-the-world
   Content: Top 100 Traditional Recipes in the World · Vori-vori · Conchitas a la parmesana · Zhēngjiǎo · Karantika · Otoro

## PHASE 3: Agent Creation

In [31]:
sys = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [32]:
from langchain.agents import create_agent

agent = create_agent(
    model = model,
    tools = [web_search],
    system_prompt = sys
)



In [33]:
#Test the agent 

from langchain.messages import HumanMessage
response = agent.invoke(
{"messages": [HumanMessage(content="I have potatos, eggs, meat, cheese and some spices, what can i make")]})

pprint(response['messages'][-1].content)

[{'extras': {'signature': 'Ct0NAb4+9vu7iHszQ7BjhIh8KTRuA1yv5EN6cb2cNhttpYMwUsZzxOaA/xvsVoSf3+0ca+x0UdzGabtHWSGfqObt1cqQO/3J2wX/rfSc+Q8dVAFSJjTdMHQax3cfludh5H1dKCYTi0vnDDYcfLHePv9iq4wSYasiRcDA9imKk5hDKN+6WbZMkxmwkCZUsBrwMl39fxbQsGD9zLccZQ8yUMOpEB75YQpNPLStLaSUHh4KcTRsTel/pvbztyMafFhYZ1dgz9wG9nqtmvbu1W+vBOPkusNVuDiHYNEG26/grOiRfOtSyBYoB+OHGRWMrbGXyOgjczd3s+Qr0j5Iydv33z4cFq7K3arvLhWXMSJy49kYBPrQDcD2+Pb5+3B+YWe3LucgmmPUX6f5VX/fjc6go1mxdWl7svlEhukge/WoaS6cbaMRC+6lfZTTeUK9pCt0up6kbBEcrdiHLe+K3IukatuQriG+EdoxmyCE3J2XzegfyLY/y+7LNpz154M7wRxCs2mVqIxPSL2JusKoSIDzK2azcL3acyrMuS1H+yax0QRraLI/XmNXXpAEwmxvQ5k7KN3harPymc+R/jHj09hAgoM7VCErC+XaCZA4FlBCNkaun4ORTyOfwU2/WVjb2OzAflkDZXwwNjk72tdo5+lvYMCLy8/yxbd1wRtFpNmMzmc8E+eBJODPys04ANK9a+vbfYHWiFe3JjfhB9af4pdV4f/ilJRz14XHdKtG3taYQW6YYP5baPv6F0R7Zo8en5ZljyLfnVT15ZqOIybyTMFq9PSSiXIEcO5n3C9g+OvwWdVFZu3bJ/eXIrF94HZetlsIw5CchVeR3QtyAs2xGUSXfINHiBiGQRSp2coB3z6MWyO/BKbcujyZAdZdHZ1HEAXGi+ib8OA5vaqpZ7I6h8J9/sBCkKxN5jSZE0fSXxqef3W2h6eY4RP7yXDbz2sjQB8EykGARKOuu/aQ/

In [34]:
pprint(response)

{'messages': [HumanMessage(content='I have potatos, eggs, meat, cheese and some spices, what can i make', additional_kwargs={}, response_metadata={}, id='f30163f7-d7bd-4f30-8201-c8908616351e'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "recipes with potatoes eggs meat cheese and spices"}'}, '__gemini_function_call_thought_signatures__': {'7e0666d6-2a4d-4d08-b1aa-ae19871ea8ae': 'CvMCAb4+9vtx8N9W4Aqq9fKAIgdWuRZahqhii90jbuWyDGRFgj6/HLYyrlB1cacJnn/1Bw5XCdcYa3jLrDI1xQW1l1nrXEXeHMD1a+d9148HXRuZX2cOuxZuGjxfp/fVd8EGSLXOXqp70Qdg0BgvInU/GWBZc/6GH0k7ynykc5qrOepYpb4JL7EzcQqN+fbi32XROQJ+mwrW3pg2OeUf5YLg+9/StFKHtxX88tMat9/c6GD4/1rD6gXnNV1FPb4lI7RKxnlMYeJGYYiL/SGNwnEtWzM3ApHuCXqcJRrKay2ly97zeiaWRTws902+vlr+PTwHPwMzAKt4EIj+FMkkB/L8s8dSeYl9q03HovPw++5lpIRgFhO/mLD06kVAvIYlSvQxSXsx4Gre5OdOMGFvb5h+gTYu5PFuPy5TSENistFwqzzsh4dJKqoDozbjbodpr1CRWxRoBJ5Wji63z6DoieX2MXVTqwYHCjpeSvwgVtxAxVLYbq8='}}, response_metadata={'finish_reason': '

In [35]:
def get_clean_response(response):
    """Extract just the text response for your users"""
    messages = response['messages']
    
    # Get the last message (the final answer)
    last_message = messages[-1]
    
    # Handle different response formats
    if isinstance(last_message.content, list):
        # Gemini format: content is a list of dicts
        for item in last_message.content:
            if item.get('type') == 'text':
                return item['text']
    elif isinstance(last_message.content, str):
        # Simple string response
        return last_message.content
    
    return "Sorry, I couldn't understand the response"

# Use it:
clean_response = get_clean_response(response)
print(clean_response)

Based on the ingredients you have (potatoes, eggs, meat, cheese, and spices), here are a few recipe suggestions:

1.  **Cheesy Potato and Egg Skillet with Meat:** You can pan-fry potatoes until crispy, then add your choice of meat (ground beef, sausage, or diced chicken would work well) and cook it through. Create wells in the mixture, crack eggs into them, sprinkle with cheese and spices, then bake or cover until the eggs are set and the cheese is melted. This is similar to a "Potato Shakshuka" but with meat.
2.  **Hearty Potato, Egg, and Meat Hash:** Dice your potatoes and meat, cook them in a skillet until browned, then scramble in the eggs and cheese at the end. Season generously with your spices.
3.  **Loaded Potato, Egg, and Meat Tacos/Burritos:** Cook diced potatoes and your chosen meat with spices. Scramble eggs separately. Warm tortillas and fill them with the potato and meat mixture, scrambled eggs, and top with cheese.
4.  **Baked Cheesy Potatoes with Eggs and Meat:** Layer 

In [36]:
# Check the second message (index 1) for tool_calls
if hasattr(response['messages'][1], 'tool_calls') and response['messages'][1].tool_calls:
    print("✅ Agent used a tool!")
    print(f"Tool used: {response['messages'][1].tool_calls[0]['name']}")
    print(f"Query: {response['messages'][1].tool_calls[0]['args']['query']}")
else:
    print("❌ Agent did NOT use a tool")

✅ Agent used a tool!
Tool used: web_search
Query: recipes with potatoes eggs meat cheese and spices


Or more simply, We look at the second message in the response:

If response['messages'][1] is an AIMessage with tool_calls → it used the tool

If response['messages'][1] is directly an AIMessage with content → it didn't

In Our first test (without system prompt), We would have seen no tool calls. In Our second test (with system prompt), We saw the tool call.



In [37]:
response2 = agent.invoke({
    "messages": [HumanMessage(content="Yes, give me the instructions for the One-Pot Spicy Eggs and Potatoes recipe please")]
})
clean_response = get_clean_response(response2)
print(clean_response)

I can provide you with instructions for One-Pot Spicy Eggs and Potatoes. Here is one recipe:

**One Pot Spicy Eggs and Potatoes Recipe**

**Ingredients:**
*   Tender, golden potatoes
*   Smoky ancho chili-spiced tomato sauce
*   Eggs
*   Kale
*   Goat cheese (optional, for topping)

**Instructions:**
1.  Prepare the smoky ancho chili-spiced tomato sauce.
2.  Add the tender, golden potatoes to the sauce.
3.  Cook until the potatoes are tender.
4.  Crack the eggs into the pot with the potatoes and sauce.
5.  Cover the pot with a lid and reduce the heat to low.
6.  Let the eggs cook gently in the barely bubbling sauce for about 10 minutes, or until desired doneness.
7.  Stir in a vibrant handful of kale towards the end of cooking, allowing it to wilt.
8.  Serve topped with creamy goat cheese, if desired.

Enjoy your One-Pot Spicy Eggs and Potatoes!


In [38]:
response2 = agent.invoke({
    "messages": [HumanMessage(content="Yes, give me the instructions for first recipe now please")]
})
clean_response = get_clean_response(response2)
print(clean_response)

I need to know what ingredients you have so I can suggest a recipe. Please tell me what you have in your kitchen!


## ==> Agents won't hold a converstation unless we use checkpointers.

In [39]:
# === PHASE 3 ENHANCEMENT: Image Input for Fridge/Ingredients ===
# 1. Upload image (works in Jupyter)
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(accept='.png,.jpg,.jpeg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload')

In [42]:
# 2. Convert uploaded image to base64
def image_to_base64(uploader):
    uploaded_file = uploader.value[0]
    img_bytes = bytes(uploaded_file["content"])
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    mime_type = uploaded_file["type"] # e.g. "image/jpeg"
    return img_b64, mime_type

# 3. Build the multimodal message (Gemini-compatible format)
def ask_agent_with_image(agent, uploader, text_prompt, config= None #we updated the function to accept  
                           # and pass the config 
                        ):
    img_b64, mime_type = image_to_base64(uploader)

    # Gemini needs this exact format for base64 images:
    question = HumanMessage(content=[
    {"type": "text", "text": text_prompt},
    {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{img_b64}"}
    }
    ])

    response = agent.invoke({"messages": [question]}, config # we updated the function to accept and 
                           #pass the config
                           )
    return get_clean_response(response)

result = ask_agent_with_image(
agent,
uploader,
"Here's a photo of my fridge. What can I cook with these ingredients?"
)
print(result)


Based on the ingredients in your fridge – potatoes, eggs, bell peppers (red and yellow), leeks, tomatoes (cherry and regular), bread, milk, and spinach – a frittata would be an excellent choice!

Here are a few frittata ideas you could make:

*   **Leek and Potato Frittata with Spinach:** This classic combination is hearty and flavorful. You can sauté the leeks and potatoes until tender, then add the spinach, and finally pour in the egg and milk mixture.
*   **Bell Pepper, Potato, and Tomato Frittata:** A colorful option using your bell peppers and tomatoes, along with potatoes and eggs.
*   **Mixed Veggie Frittata:** You could combine all the vegetables you have – potatoes, leeks, bell peppers, spinach, and tomatoes – into a delicious and nutritious frittata.

You can serve any of these frittatas with a side of your bread, perhaps toasted. You also have a cucumber, chili peppers, an orange, and a lemon, which could be used for a fresh side salad or to add a zesty touch to your meal.



## PHASE 4: Memory & Conversation

##### The problem is always true for LangChain agents — without a checkpointer, every .invoke() call starts with a blank memory.

In [43]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = model,
    tools = [web_search],
    system_prompt = sys,
    checkpointer=InMemorySaver()
    #The problem is always true for LangChain agents — 
    # without a checkpointer, every .invoke() call starts with a blank memory.
)

config = {"configurable": {"thread_id": "chef_session_1"}}

In [44]:
# Turn 1 — send image
result = ask_agent_with_image(
    agent,
    uploader,
    "Here's a photo of my fridge. What can I cook with these ingredients?",
    config=config  
)
print(result)

With the ingredients you have: potatoes, eggs, bell peppers, tomatoes, leeks, bread, and milk, here are some recipe ideas:

*   **Potato & Leek Frittata:** This is a great option for a hearty breakfast or a light dinner. You can incorporate the bell peppers and tomatoes into the frittata as well.
*   **Egg, Potato, and Tomato Breakfast/Brunch:** A simple and delicious one-pan meal that can be served with bread.
*   **Scrambled Eggs with Potatoes & Bell Peppers:** A classic and easy dish for any time of day. You can add tomatoes and leeks for extra flavor and nutrition.
*   **Spicy and Savory Tomato, Peppers, Potato, and Eggs:** If you like a bit of a kick, this recipe uses chili peppers (you have them!) along with the other ingredients.
*   **Scrambled Eggs With Peppers, Tomatoes and Potatoes:** A more refined version of scrambled eggs, often featured in "not just for breakfast" categories.

Which one would you like to know more about? I can provide you with instructions for any of the

In [45]:
# Turn 2 — agent remembers the image and previous suggestions!
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Give me the full recipe for the frittata please")]},
    config 
)
print(get_clean_response(response))

I apologize, but I am unable to access the full recipe instructions from the search result directly. However, you can find the full recipe for the "Potato & Leek Frittata" on the Wegmans website.

Here's the link to the recipe: [https://www.wegmans.com/recipes/breakfast-and-brunch/potato-leek-frittata-2737297](https://www.wegmans.com/recipes/breakfast-and-brunch/potato-leek-frittata-2737297)

It mentions that the recipe includes crispy potatoes, tender leeks, organic red bell peppers, and creamy brown eggs.


## PHASE 5: ENHANCEMENTS 

A weak point observed in last phase is the Turn 2 response, instead of giving full instructions, it just returned a link. That's because the web search tool only grabbed a snippet(short preview), not the full page content.                  
The fix is to fetch the actual page content after finding the URL 

In [46]:
# Add this alongside your web_search tool
import requests
from bs4 import BeautifulSoup

@tool
def get_page_content(url: str) -> str:
    """Fetch the full content of a webpage, useful for getting complete recipe instructions"""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Remove noise (scripts, styles, nav bars etc)
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        
        # Get clean text
        text = soup.get_text(separator="\n", strip=True)
        
        # Limit to avoid overwhelming the model
        return text[:5000]
    
    except Exception as e:
        return f"Could not fetch page: {str(e)}"

In [47]:
# Update the system prompt

sys = """
You are a personal chef. The user will give you a list of ingredients they have,
either as text or as a photo of their fridge.

You have two tools:
- web_search: use this to find recipe suggestions based on ingredients
- get_page_content: use this to fetch the FULL instructions of a specific recipe URL

When the user asks for full recipe instructions, ALWAYS use get_page_content 
on the recipe URL to retrieve the complete steps — never just return a link.
"""



In [48]:
# Update the Agent 
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[web_search, get_page_content],  # We add the new tool
    system_prompt=sys,
    checkpointer=InMemorySaver()
)

### Redo the same tests :


In [49]:
# Reset config (new agent = fresh memory)
config = {"configurable": {"thread_id": "chef_session_2"}}

In [50]:
# Turn 1 — image
result = ask_agent_with_image(
    agent,
    uploader,
    "Here's a photo of my fridge. What can I cook with these ingredients?",
    config=config
)
print(result)

With the ingredients you have, you could make a delicious **Potato, Spinach, Bell Pepper, and Leek Frittata**! This would use your potatoes, spinach, bell peppers, leek, and eggs. You could also add some of the chili peppers for a bit of a kick.

On the side, you could prepare a fresh salad with your cherry tomatoes and cucumber. A simple dressing could be made with lemon juice.

You also have bread which would be great with the frittata, and an orange for a fresh fruit snack or juice.

Would you like the full recipe for the Potato, Spinach, Bell Pepper, Leek and Basil Frittata from justfarmed.com?


In [51]:
# Turn 2 — should now return FULL instructions, not just a link
response = agent.invoke(
    {"messages": [HumanMessage(content="Give me the full recipe for the frittata please")]},
    config
)
print(get_clean_response(response))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.500746397s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

# We hit Gemini's free tier daily limit of 20 requests, We wait until tomorrow